# Robustness Evaluation for GNN Fraud Detection (Elliptic)

This notebook runs **clean training + adversarial robustness experiments** for each model (GCN, GAT, GraphSAGE), collects metrics from `attacks/*/config.json` and `attacks/*/metrics.json`, and produces **paper-ready plots + tables**.

**Attacks supported by the repo (gpt6):** FGSM, PGD, Nettack-Local, Node Injection, MonTi-style injection.

---

## 0) Configure the experiment
Edit the parameters below, then run the notebook top-to-bottom.


In [ ]:

#@title Experiment configuration
REPO_MODE = "upload_zip"  #@param ["upload_zip", "git_clone"]
REPO_ZIP_NAME = "gnn-adversarial-blockchain-gpt6.zip"  #@param {type:"string"}
REPO_URL = ""  #@param {type:"string"}
REPO_DIRNAME = "gnn-adversarial-blockchain-gpt"  # repo folder name after unzip/clone

# Data: expects data/processed/elliptic/data.pt
DATA_PT_PATH = "data/processed/elliptic/data.pt"  # relative to repo root

# Which models to train/evaluate
MODELS = ["gcn", "gat", "graphsage"]

# Training
TRAIN_IF_MISSING = True
TRAIN_EPOCHS_OVERRIDE = None  # set an int to override epochs in training scripts

# Attack sweep: attack-selection randomness (targets sampling)
ATTACK_SEEDS = [0, 1, 2]

# Target selection knobs (used by most run_* scripts)
SPLIT = "test"
ATTACK_ONLY_ILLICIT = True
ONLY_CLEAN_CORRECT = True
ATTACK_FRACTION = 0.02

# Output
ARTIFACTS_DIR = "artifacts"  # saved plots/tables here


## 1) Setup & install dependencies


In [ ]:

import os, sys, json, shutil, glob, subprocess, textwrap, time
from pathlib import Path

def run(cmd, cwd=None):
    print("
$", " ".join(cmd))
    subprocess.check_call(cmd, cwd=cwd)

# Show GPU
try:
    import torch
    print("torch:", torch.__version__, "cuda:", torch.version.cuda, "gpu:", torch.cuda.is_available())
except Exception as e:
    print("torch not ready yet:", e)


In [ ]:

# Install PyTorch Geometric stack robustly on Colab
# (Works with the preinstalled torch in Colab; avoids forcing a torch reinstall.)

import re
import sys
import subprocess

def install_pyg():
    import torch
    tv = torch.__version__.split('+')[0]
    cuda = torch.version.cuda

    # Colab CUDA wheels commonly use cu118/cu121. If cuda is None, install CPU wheels.
    if cuda is None:
        whl = f"https://data.pyg.org/whl/torch-{tv}+cpu.html"
    else:
        cu = cuda.replace('.', '')
        whl = f"https://data.pyg.org/whl/torch-{tv}+cu{cu}.html"

    # Core deps
    pkgs = [
        "pyg_lib",
        "torch_scatter",
        "torch_sparse",
        "torch_cluster",
        "torch_spline_conv",
        "torch_geometric",
    ]

    print("Installing PyG wheels from:", whl)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs + ["-f", whl])

# Base deps
run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "pandas", "matplotlib", "scikit-learn", "tqdm"]) 

# Install PyG
install_pyg()

print("Done.")


## 2) Get the repo (upload zip OR git clone)


In [ ]:

from google.colab import files

ROOT = Path.cwd()

if REPO_MODE == "upload_zip":
    if not Path(REPO_ZIP_NAME).exists():
        print("Upload your repo zip:")
        uploaded = files.upload()
        assert REPO_ZIP_NAME in uploaded, f"Expected {REPO_ZIP_NAME}"

    # unzip into /content
    if Path(REPO_DIRNAME).exists():
        shutil.rmtree(REPO_DIRNAME)
    run(["bash", "-lc", f"unzip -q {REPO_ZIP_NAME}"])

    # Some zips contain a parent folder; try to locate the repo dir
    if not Path(REPO_DIRNAME).exists():
        # pick the first folder containing 'src'
        candidates = [p for p in Path('.').glob('**/src')]
        if not candidates:
            raise RuntimeError("Could not find repo folder with src/")
        REPO_DIRNAME = str(candidates[0].parent)

elif REPO_MODE == "git_clone":
    assert REPO_URL, "Set REPO_URL"
    if Path(REPO_DIRNAME).exists():
        shutil.rmtree(REPO_DIRNAME)
    run(["git", "clone", REPO_URL, REPO_DIRNAME])

REPO_ROOT = Path(REPO_DIRNAME).resolve()
print("Repo root:", REPO_ROOT)

# Install requirements (repo-specific)
req = REPO_ROOT / "requirements.txt"
if req.exists():
    run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])

# Add repo to path
sys.path.insert(0, str(REPO_ROOT))


## 3) Ensure the processed dataset exists (`data/processed/elliptic/data.pt`)

Your repo expects:

- `data/processed/elliptic/data.pt`

If you don't have it on Colab yet, upload it in the cell below.


In [ ]:

from google.colab import files

data_pt = REPO_ROOT / DATA_PT_PATH
if not data_pt.exists():
    print("Missing:", data_pt)
    print("Please upload data.pt now (it will be placed into data/processed/elliptic/).")

    uploaded = files.upload()  # upload a file named data.pt
    assert "data.pt" in uploaded, "Upload a file named data.pt"

    data_pt.parent.mkdir(parents=True, exist_ok=True)
    shutil.move("data.pt", str(data_pt))
    print("Saved:", data_pt)
else:
    print("Found:", data_pt)


## 4) Train models (if checkpoints are missing)

Each training script saves:
- `models/<model>_YYYYMMDD_HHMMSS/model.pt`

Attack scripts load the **latest** run per model.


In [ ]:

def latest_run_dir(model_name: str) -> Path:
    mdir = REPO_ROOT / "models"
    runs = sorted([p for p in mdir.glob(f"{model_name}_*") if (p / "model.pt").exists()])
    return runs[-1] if runs else None

def maybe_train(model_name: str):
    run_dir = latest_run_dir(model_name)
    if run_dir is not None:
        print(f"✓ {model_name}: checkpoint exists -> {run_dir}")
        return
    if not TRAIN_IF_MISSING:
        raise RuntimeError(f"No checkpoint for {model_name} and TRAIN_IF_MISSING=False")

    script = {
        "gcn": "scripts/train_gcn.py",
        "gat": "scripts/train_gat.py",
        "graphsage": "scripts/train_graphsage.py",
    }[model_name]

    print(f"Training {model_name} ...")

    # Optional: override epochs by patching CONFIG in-script (simple text replacement).
    # If you want this, set TRAIN_EPOCHS_OVERRIDE.
    if TRAIN_EPOCHS_OVERRIDE is not None:
        import re
        path = REPO_ROOT / script
        txt = path.read_text(encoding="utf-8")
        txt2 = re.sub(r'("epochs"\s*:\s*)\d+', r"\g<1>%d" % int(TRAIN_EPOCHS_OVERRIDE), txt)
        path.write_text(txt2, encoding="utf-8")
        print("Patched epochs ->", TRAIN_EPOCHS_OVERRIDE)

    run([sys.executable, str(REPO_ROOT / script)], cwd=str(REPO_ROOT))
    run_dir = latest_run_dir(model_name)
    print(f"✓ {model_name}: trained -> {run_dir}")

for m in MODELS:
    maybe_train(m)


## 5) Add a sweep runner (writes `attacks/results_summary.csv`)

This creates `scripts/run_sweep.py` in the repo.


In [ ]:

SWEEP_PATH = REPO_ROOT / "scripts" / "run_sweep.py"
if not SWEEP_PATH.exists():
    SWEEP_PATH.write_text(textwrap.dedent('''
    import os
    import sys
    import json
    import csv
    import time
    import itertools
    import importlib.util

    import yaml

    ATTACK_TO_SCRIPT = {
        "fgsm": "scripts/run_fgsm_attack.py",
        "pgd": "scripts/run_pgd_attack.py",
        "nettack": "scripts/run_nettack_attack.py",
        "node_injection": "scripts/run_node_injection_attack.py",
        "monti": "scripts/run_monti_attack.py",
    }

    def repo_root() -> str:
        return os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))

    def attacks_root() -> str:
        return os.path.join(repo_root(), "attacks")

    def load_script_module(name: str, path: str):
        spec = importlib.util.spec_from_file_location(name, path)
        if spec is None or spec.loader is None:
            raise RuntimeError(f"Failed to load module from {path}")
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        return mod

    def cartesian(params):
        if not params:
            return [{}]
        keys = list(params.keys())
        values = [params[k] if isinstance(params[k], list) else [params[k]] for k in keys]
        out = []
        for prod in itertools.product(*values):
            out.append({k: v for k, v in zip(keys, prod)})
        return out

    def list_attack_dirs():
        root = attacks_root()
        if not os.path.isdir(root):
            return []
        return [os.path.join(root, d) for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))]

    def newest_dir(created_after, prev_set):
        cand = []
        for d in list_attack_dirs():
            if d in prev_set:
                continue
            try:
                mt = os.path.getmtime(d)
            except Exception:
                continue
            if mt >= created_after - 1e-6:
                cand.append((mt, d))
        if not cand:
            all_dirs = [(os.path.getmtime(d), d) for d in list_attack_dirs()]
            if not all_dirs:
                raise RuntimeError("No attacks/* directory found after run.")
            all_dirs.sort(key=lambda x: x[0])
            return all_dirs[-1][1]
        cand.sort(key=lambda x: x[0])
        return cand[-1][1]

    def flatten(obj, prefix=""):
        out = {}
        if isinstance(obj, dict):
            for k, v in obj.items():
                key = f"{prefix}.{k}" if prefix else str(k)
                out.update(flatten(v, key))
        elif isinstance(obj, list):
            out[prefix] = json.dumps(obj, ensure_ascii=False)
        else:
            out[prefix] = obj
        return out

    def read_json(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)

    def summarize_attacks_to_csv(out_csv):
        rows = []
        for d in sorted(list_attack_dirs()):
            cfg_path = os.path.join(d, "config.json")
            met_path = os.path.join(d, "metrics.json")
            if not (os.path.exists(cfg_path) and os.path.exists(met_path)):
                continue

            cfg = read_json(cfg_path)
            met = read_json(met_path)

            row = {"run_dir": os.path.relpath(d, repo_root())}
            row.update({f"config.{k}": v for k, v in flatten(cfg).items()})
            row.update({f"metrics.{k}": v for k, v in flatten(met).items()})
            rows.append(row)

        if not rows:
            print("No runs found under attacks/* with config.json + metrics.json.")
            return

        cols = set()
        for r in rows:
            cols |= set(r.keys())
        cols = sorted(cols)

        os.makedirs(os.path.dirname(out_csv), exist_ok=True)
        with open(out_csv, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=cols)
            w.writeheader()
            for r in rows:
                w.writerow(r)

        print(f"Wrote summary: {os.path.relpath(out_csv, repo_root())} (rows={len(rows)})")

    def apply_globals(mod, overrides):
        for k, v in overrides.items():
            setattr(mod, k, v)

    def normalize_pgd_params(grid):
        if "ALPHA" in grid and isinstance(grid["ALPHA"], str) and grid["ALPHA"].lower() == "auto":
            eps = float(grid["EPS"])
            steps = int(grid["STEPS"])
            grid["ALPHA"] = 2.0 * eps / max(1, steps)
        return grid

    def run_one(mod, overrides):
        prev = set(list_attack_dirs())
        start = time.time()
        apply_globals(mod, overrides)

        for attempt in range(3):
            try:
                mod.main()
                break
            except FileExistsError:
                time.sleep(0.35)
                if attempt == 2:
                    raise
            finally:
                time.sleep(0.25)

        run_dir = newest_dir(start, prev)
        return run_dir

    def main():
        root = repo_root()
        cfg_path = sys.argv[1] if len(sys.argv) > 1 else os.path.join(root, "config", "experiments.yaml")
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = yaml.safe_load(f) or {}

        g = cfg.get("global", {}) or {}
        sweeps = cfg.get("sweeps", []) or []

        models = g.get("models", ["gcn"])
        seeds = g.get("seeds", [0])

        g_split = g.get("split", "test")
        g_only_illicit = bool(g.get("attack_only_illicit", True))
        g_only_clean_correct = bool(g.get("only_clean_correct", True))
        g_attack_fraction = float(g.get("attack_fraction", 0.02))

        modules = {}
        for attack, rel in ATTACK_TO_SCRIPT.items():
            path = os.path.join(root, rel)
            modules[attack] = load_script_module(f"_run_{attack}", path)

        for sweep in sweeps:
            attack = str(sweep["attack"]).lower()
            if attack not in modules:
                raise ValueError(f"Unknown attack '{attack}'")

            param_grid = sweep.get("params", {}) or {}
            combos = cartesian(param_grid)

            for model_name in models:
                for seed in seeds:
                    for combo in combos:
                        overrides = {
                            "MODEL_NAME": model_name,
                            "SPLIT": g_split,
                            "SEED": int(seed),
                            "ATTACK_ONLY_ILLICIT": g_only_illicit,
                            "ONLY_CLEAN_CORRECT": g_only_clean_correct,
                            "ATTACK_FRACTION": g_attack_fraction,
                        }
                        overrides.update(combo)
                        if attack == "pgd":
                            overrides = normalize_pgd_params(overrides)

                        mod = modules[attack]
                        filtered = {k: v for k, v in overrides.items() if hasattr(mod, k)}
                        run_dir = run_one(mod, filtered)
                        print(f"Done: {attack} {model_name} seed={seed} -> {os.path.relpath(run_dir, root)}")

        summarize_attacks_to_csv(os.path.join(attacks_root(), "results_summary.csv"))

    if __name__ == "__main__":
        main()
    '''), encoding="utf-8")
    print("Created:", SWEEP_PATH)
else:
    print("Found:", SWEEP_PATH)


## 6) Define the sweep (budgets)

These defaults are a solid starting grid. You can edit them.


In [ ]:

SWEEP = {
    "fgsm": {
        "EPS": [0.01, 0.03, 0.05, 0.1],
    },
    "pgd": {
        "EPS": [0.03, 0.05],
        "STEPS": [10, 30],
        "ALPHA": ["auto"],
        "RANDOM_START": [True],
    },
    "nettack": {
        "N_PERTURBATIONS": [1, 3, 5, 8],
        "SAMPLE_SIZE": [50, 200],
        "ATTACK_INCOMING": [True],
        "EARLY_STOP": [True],
        # ATTACK_ONLY_ILLICIT / ATTACK_FRACTION are controlled globally
    },
    "node_injection": {
        "N_INJECT": [1, 3],
        "EDGES_PER_INJECTED": [1, 5],
        "EPS": [0.03, 0.05],
        "STEPS": [30],
    },
    "monti": {
        "N_INJECT": [5, 10],
        "EDGE_BUDGET": [20, 50, 100],
        "K_HOP": [1, 2],
        "EPS": [0.05],
    },
}


## 7) Run experiments **separately for each model**

This will run attack scripts and create many folders under `attacks/`.


In [ ]:

import yaml

def write_yaml_for_model(model_name: str, out_path: Path):
    sweeps = []
    for attack, params in SWEEP.items():
        sweeps.append({"attack": attack, "params": params})

    cfg = {
        "global": {
            "split": SPLIT,
            "attack_only_illicit": bool(ATTACK_ONLY_ILLICIT),
            "only_clean_correct": bool(ONLY_CLEAN_CORRECT),
            "attack_fraction": float(ATTACK_FRACTION),
            "seeds": [int(s) for s in ATTACK_SEEDS],
            "models": [model_name],
        },
        "sweeps": sweeps,
    }
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
    return out_path

for model_name in MODELS:
    cfg_path = REPO_ROOT / "config" / f"experiments_{model_name}.yaml"
    write_yaml_for_model(model_name, cfg_path)

    print("
=== Running sweeps for", model_name, "===")
    run([sys.executable, str(REPO_ROOT / "scripts" / "run_sweep.py"), str(cfg_path)], cwd=str(REPO_ROOT))


## 8) Load results into a DataFrame


In [ ]:

import pandas as pd

SUMMARY_CSV = REPO_ROOT / "attacks" / "results_summary.csv"
assert SUMMARY_CSV.exists(), "Missing attacks/results_summary.csv"

df = pd.read_csv(SUMMARY_CSV)
print("rows:", len(df), "cols:", len(df.columns))

df.head(3)


## 9) Utility: normalize columns + compute PR-AUC

Some plots use budget columns like `eps`, `k`, `edge_budget`. This cell extracts them from config.


In [ ]:
import numpy as np

# Helper: get column if exists, else NaN

def col(df, name):
    return df[name] if name in df.columns else np.nan

def first_non_nan(*arrs):
    """Elementwise first-non-NaN across arrays (pandas Series or numpy)."""
    out = None
    for a in arrs:
        if isinstance(a, float) and np.isnan(a):
            continue
        if out is None:
            out = a
        else:
            out = out.where(~out.isna(), a) if hasattr(out, 'isna') else np.where(np.isnan(out), a, out)
    if out is None:
        return np.nan
    return out

out = df.copy()

# Identify model + attack
out["model"] = col(out, "config.model_name")
out["attack"] = col(out, "config.attack")

# Budgets per attack (not all attacks have all fields)
out["eps"] = col(out, "config.attack_params.eps")
out["pgd_steps"] = col(out, "config.attack_params.steps")
out["k"] = col(out, "config.attack_params.n_perturbations")
out["sample_size"] = col(out, "config.attack_params.sample_size")
out["n_inject"] = first_non_nan(
    col(out, "config.attack_params.n_inject"),
    col(out, "config.budgets.n_inject"),
)
out["edges_per_injected"] = col(out, "config.attack_params.edges_per_injected")
out["edge_budget"] = first_non_nan(
    col(out, "config.attack_params.edge_budget"),
    col(out, "config.budgets.edge_budget"),
)
out["k_hop"] = first_non_nan(
    col(out, "config.attack_params.k_hop"),
    col(out, "config.candidates.K_hop"),
)

# Core split metrics (FGSM/PGD/Injection/MoNTi record these; NettackLocal may not)
out["f1_pos_clean"] = col(out, "metrics.f1.pos_clean")
out["f1_pos_adv"] = col(out, "metrics.f1.pos_adv")
out["f1_macro_clean"] = col(out, "metrics.f1.macro_clean")
out["f1_macro_adv"] = col(out, "metrics.f1.macro_adv")
out["roc_auc_clean"] = col(out, "metrics.roc_auc.clean")
out["roc_auc_adv"] = col(out, "metrics.roc_auc.adv")

# ASR has multiple schemas across scripts
# - FGSM/PGD: metrics.asr.value and metrics.asr_pos_neg.*
# - NettackLocal: metrics.asr (scalar)
# - NodeInjection / MonTi: metrics.asr.attacked.value and metrics.asr.pos_neg_on_attacked.*
out["asr"] = first_non_nan(
    col(out, "metrics.asr.value"),
    col(out, "metrics.asr"),
    col(out, "metrics.asr.attacked.value"),
)

out["asr_pos"] = first_non_nan(
    col(out, "metrics.asr_pos_neg.asr_pos"),
    col(out, "metrics.asr.pos_neg_on_attacked.asr_pos"),
    # fallback: if missing (e.g., NettackLocal), use asr as a proxy
    out["asr"],
)

out["asr_neg"] = first_non_nan(
    col(out, "metrics.asr_pos_neg.asr_neg"),
    col(out, "metrics.asr.pos_neg_on_attacked.asr_neg"),
)

# Confidence drop has two schemas
out["conf_drop"] = first_non_nan(
    col(out, "metrics.mean_confidence_drop.value"),
    col(out, "metrics.mean_confidence_drop_clean_correct.value"),
)

# Drops
out["f1_pos_drop"] = out["f1_pos_clean"] - out["f1_pos_adv"]
out["roc_auc_drop"] = out["roc_auc_clean"] - out["roc_auc_adv"]

# Save normalized
norm_path = REPO_ROOT / "attacks" / "results_normalized.csv"
out.to_csv(norm_path, index=False)
print("Saved:", norm_path)

out.head(3)


## 10) Paper-ready plots

For each model, this generates:
- F1_pos vs budget (FGSM/PGD)
- ASR_pos vs budget (Nettack)
- Injection robustness (Node injection, MonTi)

Saved under `artifacts/<model>/plots/` as PNG (300 dpi) + PDF.


In [ ]:

import matplotlib.pyplot as plt

def savefig(path_base: Path):
    path_base.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(str(path_base.with_suffix('.png')), dpi=300, bbox_inches='tight')
    plt.savefig(str(path_base.with_suffix('.pdf')), bbox_inches='tight')

def plot_curve(df_sub, x, y, group_cols=None, title="", xlabel="", ylabel=""):
    group_cols = group_cols or []
    df_sub = df_sub.dropna(subset=[x, y])
    if df_sub.empty:
        return

    # aggregate by (x + group_cols) over attack seeds
    keys = [x] + group_cols
    agg = df_sub.groupby(keys)[y].agg(['mean', 'std', 'count']).reset_index()

    plt.figure()
    if group_cols:
        for gvals, gdf in agg.groupby(group_cols):
            label = ",".join(f"{c}={v}" for c, v in zip(group_cols, (gvals if isinstance(gvals, tuple) else (gvals,))))
            gdf = gdf.sort_values(x)
            plt.plot(gdf[x], gdf['mean'], marker='o', label=label)
            if (gdf['count'] > 1).any():
                plt.fill_between(gdf[x], gdf['mean']-gdf['std'].fillna(0), gdf['mean']+gdf['std'].fillna(0), alpha=0.2)
    else:
        agg = agg.sort_values(x)
        plt.plot(agg[x], agg['mean'], marker='o')
        if (agg['count'] > 1).any():
            plt.fill_between(agg[x], agg['mean']-agg['std'].fillna(0), agg['mean']+agg['std'].fillna(0), alpha=0.2)

    plt.title(title)
    plt.xlabel(xlabel or x)
    plt.ylabel(ylabel or y)
    plt.grid(True)
    if group_cols:
        plt.legend()

def make_plots_for_model(model_name: str, df_all: pd.DataFrame):
    dfm = df_all[df_all['model'] == model_name].copy()
    out_dir = REPO_ROOT / ARTIFACTS_DIR / model_name / "plots"

    # FGSM: F1_pos vs eps
    d = dfm[dfm['attack'].str.contains('FGSM', na=False)]
    plot_curve(d, 'eps', 'f1_pos_adv', title=f"{model_name.upper()} | FGSM | F1_pos vs eps", xlabel='eps', ylabel='F1_pos (adv)')
    savefig(out_dir / "fgsm_f1pos_vs_eps")

    # PGD: F1_pos vs eps, grouped by steps
    d = dfm[dfm['attack'].str.contains('PGD', na=False)]
    plot_curve(d, 'eps', 'f1_pos_adv', group_cols=['pgd_steps'], title=f"{model_name.upper()} | PGD | F1_pos vs eps", xlabel='eps', ylabel='F1_pos (adv)')
    savefig(out_dir / "pgd_f1pos_vs_eps")

    # Nettack: ASR_pos vs k, grouped by sample_size
    d = dfm[dfm['attack'].str.contains('Nettack', na=False)]
    plot_curve(d, 'k', 'asr_pos', group_cols=['sample_size'], title=f"{model_name.upper()} | Nettack | ASR_pos vs k", xlabel='k (perturbations)', ylabel='ASR_pos')
    savefig(out_dir / "nettack_asrpos_vs_k")

    # Node injection: ASR_pos vs edges_per_injected (for fixed n_inject)
    d = dfm[dfm['attack'].str.contains('NodeInjection', na=False)]
    plot_curve(d, 'edges_per_injected', 'asr_pos', group_cols=['n_inject'], title=f"{model_name.upper()} | Node Injection | ASR_pos", xlabel='edges_per_injected', ylabel='ASR_pos')
    savefig(out_dir / "nodeinj_asrpos")

    # MonTi: ASR_pos vs edge_budget
    d = dfm[dfm['attack'].str.contains('MonTi', na=False)]
    plot_curve(d, 'edge_budget', 'asr_pos', group_cols=['n_inject'], title=f"{model_name.upper()} | MonTi | ASR_pos vs edge_budget", xlabel='edge_budget', ylabel='ASR_pos')
    savefig(out_dir / "monti_asrpos_vs_edgebudget")

    print("Saved plots ->", out_dir)

for m in MODELS:
    make_plots_for_model(m, out)


## 11) Paper-ready tables (CSV + LaTeX)

Creates:
- `artifacts/<model>/tables/clean_metrics.csv`
- `artifacts/<model>/tables/robustness_summary.csv`
- `artifacts/<model>/tables/*.tex`

The robustness table reports the **mean over attack seeds** for each budget.


In [ ]:

from pandas import DataFrame

def to_latex_table(df: DataFrame, path: Path, caption: str, label: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    tex = df.to_latex(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, (float, np.floating)) else str(x))
    tex = tex.replace('\toprule', f"\toprule
\caption{{{caption}}}\label{{{label}}}\\")
    path.write_text(tex, encoding='utf-8')


def make_tables_for_model(model_name: str, df_all: pd.DataFrame):
    dfm = df_all[df_all['model'] == model_name].copy()
    tdir = REPO_ROOT / ARTIFACTS_DIR / model_name / "tables"
    tdir.mkdir(parents=True, exist_ok=True)

    # Clean metrics: take the first row per attack (they all store clean metrics), so deduplicate by model.
    clean = dfm[["model", "f1_pos_clean", "f1_macro_clean", "roc_auc_clean"]].dropna().drop_duplicates(subset=["model"]).head(1)
    clean.to_csv(tdir / "clean_metrics.csv", index=False)
    to_latex_table(clean, tdir / "clean_metrics.tex", caption=f"Clean performance for {model_name.upper()}.", label=f"tab:{model_name}:clean")

    # Robustness summary: per attack + budget, average over seeds
    cols = ["attack", "eps", "pgd_steps", "k", "sample_size", "n_inject", "edges_per_injected", "edge_budget", "f1_pos_adv", "f1_pos_drop", "roc_auc_adv", "asr_pos", "conf_drop"]
    base = dfm[cols].copy()

    # Grouping keys vary by attack; keep all and group by non-null keys.
    # Simpler: group by (attack, eps, pgd_steps, k, sample_size, n_inject, edges_per_injected, edge_budget)
    gkeys = ["attack", "eps", "pgd_steps", "k", "sample_size", "n_inject", "edges_per_injected", "edge_budget"]
    grp = base.groupby(gkeys, dropna=False).agg({
        "f1_pos_adv": "mean",
        "f1_pos_drop": "mean",
        "roc_auc_adv": "mean",
        "asr_pos": "mean",
        "conf_drop": "mean",
    }).reset_index()

    # Make table more readable: sort by attack then by budget-ish columns
    grp = grp.sort_values(["attack", "eps", "pgd_steps", "k", "edge_budget", "n_inject", "edges_per_injected", "sample_size"], na_position='last')

    grp.to_csv(tdir / "robustness_summary.csv", index=False)
    to_latex_table(grp.head(40), tdir / "robustness_summary_head.tex",
                   caption=f"Robustness summary (head) for {model_name.upper()}. Full table in CSV.",
                   label=f"tab:{model_name}:robust_head")

    print("Saved tables ->", tdir)

for m in MODELS:
    make_tables_for_model(m, out)


## 12) Export artifacts

This zips `artifacts/` so you can download and directly insert figures/tables into your paper & slides.


In [ ]:

import zipfile
from google.colab import files

zip_path = REPO_ROOT / f"{ARTIFACTS_DIR}.zip"
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in (REPO_ROOT / ARTIFACTS_DIR).rglob('*'):
        if p.is_file():
            z.write(p, arcname=str(p.relative_to(REPO_ROOT)))

print("Created:", zip_path)
files.download(str(zip_path))
